# Agenda reuniones — Qwen3-4B-Instruct-2507 en Colab (repo privado)

Skill portable desde `.opencode/skills/agenda-reuniones/SKILL.md` + HF Inference API + Google Calendar.
Zona fija: `America/Santiago`.

**Pre-requisitos en Colab → icono llave Secrets:**
- `HF_TOKEN` (huggingface.co/settings/tokens, Read)
- `GITHUB_PAT` (github Settings → Developer settings → Tokens classic, scope `repo`) — solo si hay que clonar el privado.
- `token.json` de Calendar subido a Drive (generado en local con `python scripts/get_google_token.py`).


In [ ]:
!pip install -q huggingface_hub google-api-python-client google-auth-oauthlib pytz dateparser

In [ ]:
import pathlib

SKILL_PATHS = [
    pathlib.Path('.opencode/skills/agenda-reuniones/SKILL.md'),
    pathlib.Path('SML-GEN-AI-for-Calendar-services/.opencode/skills/agenda-reuniones/SKILL.md'),
]
skill_text = None
for p in SKILL_PATHS:
    if p.exists():
        skill_text = p.read_text()
        print(f'SKILL OK: {p}')
        break

if skill_text is None:
    # No estamos dentro del repo -> clonar privado con PAT
    from google.colab import userdata
    import os
    PAT = userdata.get('GITHUB_PAT')
    assert PAT, 'Falta Secret GITHUB_PAT para clonar el repo privado'
    !git clone https://{PAT}@github.com/maty20arriagada/SML-GEN-AI-for-Calendar-services.git
    p = pathlib.Path('SML-GEN-AI-for-Calendar-services/.opencode/skills/agenda-reuniones/SKILL.md')
    skill_text = p.read_text()
    print(f'SKILL OK tras clone: {p}')

print(skill_text[:800])

In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient
import json

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Falta Secret HF_TOKEN'

client = InferenceClient(model='Qwen/Qwen3-4B-Instruct-2507', token=HF_TOKEN)
SYSTEM = skill_text + '\nResponde SOLO JSON: {"titulo": str, "fecha_txt": str, "duracion_min": int}.'

def parse_frase(frase: str):
    prompt = f'{SYSTEM}\n\nFrase: {frase}\nJSON:'
    out = client.text_generation(prompt, max_new_tokens=256, temperature=0.2)
    # Qwen 4B a veces envuelve en ```json
    out = out.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    return json.loads(out)

# Test sin Calendar:
print(parse_frase('Reunión con Ana mañana a las 15:00'))

In [ ]:
import dateparser, datetime, pytz
TZ = pytz.timezone('America/Santiago')

def normalizar(fecha_txt: str, duracion_min: int = 60):
    dt = dateparser.parse(fecha_txt, settings={'TIMEZONE': 'America/Santiago', 'RETURN_AS_TIMEZONE_AWARE': True, 'PREFER_DATES_FROM': 'future'})
    assert dt, f'No pude interpretar fecha: {fecha_txt}'
    if dt.tzinfo is None:
        dt = TZ.localize(dt)
    else:
        dt = dt.astimezone(TZ)
    end = dt + datetime.timedelta(minutes=duracion_min)
    return dt.isoformat(), end.isoformat()

print(normalizar('mañana a las 15:00', 60))

In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
import pathlib

def get_calendar():
    # 1) token.json en Drive (recomendado) 2) subido a /content
    cands = [pathlib.Path('/content/drive/MyDrive/token.json'), pathlib.Path('token.json'), pathlib.Path('SML-GEN-AI-for-Calendar-services/token.json')]
    tok = next((p for p in cands if p.exists()), None)
    if tok is None:
        # monta Drive una vez
        from google.colab import drive
        drive.mount('/content/drive')
        tok = pathlib.Path('/content/drive/MyDrive/token.json')
    assert tok.exists(), 'Sube token.json a Drive con python scripts/get_google_token.py en local'
    creds = Credentials.from_authorized_user_file(str(tok), ['https://www.googleapis.com/auth/calendar.events'])
    return build('calendar', 'v3', credentials=creds)

def crear_evento(titulo: str, start_iso: str, end_iso: str, descripcion: str = ''):
    svc = get_calendar()
    body = {'summary': titulo, 'description': descripcion,
            'start': {'dateTime': start_iso, 'timeZone': 'America/Santiago'},
            'end': {'dateTime': end_iso, 'timeZone': 'America/Santiago'}}
    ev = svc.events().insert(calendarId='primary', body=body).execute()
    return ev.get('id'), ev.get('htmlLink')


In [ ]:
frase = 'Reunión con Ana mañana a las 15:00'  # cámbiala
parsed = parse_frase(frase)
start_iso, end_iso = normalizar(parsed['fecha_txt'], int(parsed.get('duracion_min', 60)))
eid, link = crear_evento(parsed['titulo'], start_iso, end_iso, frase)
print(f'OK {eid} {link} {start_iso} America/Santiago')